# 实验 4.7-1 复核：PnP-ULA 链长对照验证（Colab 一键脚本）

**目的**：用同一份算法、同一个随机种子与同一份含噪观测 $y$，仅改变**迭代长度 / 核形式**，验证复核结论：

1. 现实验（500 次迭代、简化无投影核）的结果 ≈ 官方参考链 burn-in 边界（第 500 步）的瞬时状态，**并非收敛结果**；
2. 跑满官方 10 000 次后，PSNR 应达到 **≈ 33.2 dB**、SSIM **≈ 0.921**、NRMSE **≈ 0.031**，后验方差图平坦区应转黑，ESS 明显上升；
3. 官方默认的**投影 PPnP-ULA** 核与本实验的**简化无投影核**在 10 000 次下差异很小（投影基本不被触发）。

| 组别 | maxit | burn-in | 存样 | Markov 核 | 预计 GPU 耗时 |
|---|---|---|---|---|---|
| A 短链（复现现实验） | 500 | 50 | 50（间隔 9） | 简化无投影（= 4.7-1.py） | ≈ 0.5 分钟 |
| B 长链·简化核 | 10 000 | 500 | 2 000（间隔 5） | 简化无投影 | ≈ 8 分钟 |
| C 长链·官方核 | 10 000 | 500 | 2 000（间隔 5） | 投影 PPnP-ULA（= lab2 官方） | ≈ 8 分钟 |

**使用方法**：菜单 `运行时 → 更改运行时类型 → GPU → 保存`，然后 `代码执行程序 → 全部运行`。
素材（cman.png / sampling_tools / 预训练模型）优先读取 Google Drive 中的 `MyDrive/实验4.7-1`；
若不存在则自动从官方仓库 `github.com/MI2G/sampling-tutorials` 克隆，无需手动上传。
结果保存在 `/content/verify_outputs/`，若挂载了 Drive 会同步一份到 `MyDrive/实验4.7-1/verify_outputs/`。

In [ ]:
# ① 环境准备：安装统计诊断库（Colab 自带 torch / skimage / matplotlib）
import sys, subprocess
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arviz', 'statsmodels'], check=False)

import torch
print('torch =', torch.__version__, '| GPU 可用 =', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit('未检测到 GPU：请先在「运行时 → 更改运行时类型」中选择 GPU，再重新全部运行。'
                     '（CPU 上两组长链各需约 5.5 小时）')

In [ ]:
# ② 定位实验素材：优先 Drive 中的实验目录，否则自动克隆官方仓库
import os, sys, subprocess

MOUNT_DRIVE = True          # 不需要 Drive 可改成 False，结果只保存在 /content
DRIVE_EXP = '/content/drive/MyDrive/实验4.7-1'
REPO_DIR  = '/content/sampling-tutorials'

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive 未挂载，将改用 GitHub 素材：', repr(e))

EXP_DIR = None
for c in (DRIVE_EXP, REPO_DIR):
    if os.path.isfile(os.path.join(c, 'cman.png')):
        EXP_DIR = c
        break

if EXP_DIR is None:
    print('未找到本地素材，正在克隆 https://github.com/MI2G/sampling-tutorials ...')
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/MI2G/sampling-tutorials.git', REPO_DIR], check=True)
    EXP_DIR = REPO_DIR

MODEL_PATH = os.path.join(EXP_DIR, 'Pretrained_models', 'RealSN_DnCNN_noise5.pth')
assert os.path.isfile(MODEL_PATH), f'模型缺失: {MODEL_PATH}'
sys.path.insert(0, EXP_DIR)
print('素材目录 =', EXP_DIR)
print('模型文件 =', MODEL_PATH, f'({os.path.getsize(MODEL_PATH)/1e6:.1f} MB)')

# 输出目录：/content 一份；Drive 实验目录下再同步一份
OUT_DIRS = ['/content/verify_outputs']
if os.path.isdir(DRIVE_EXP):
    OUT_DIRS.append(os.path.join(DRIVE_EXP, 'verify_outputs'))
for _d in OUT_DIRS:
    os.makedirs(_d, exist_ok=True)
print('输出目录 =', OUT_DIRS)

In [ ]:
# ③ 导入依赖；中文字体存在则用中文图题，否则自动切换英文（不影响结果）
import math, time, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from sampling_tools.blur_operators import blur_operators
from sampling_tools.load_model import load_model
from sampling_tools.welford import welford
from sampling_tools.measures import PSNR, SSIM, NRMSE
from statsmodels.graphics.tsaplots import plot_acf
import arviz

CN = False
try:
    sys.path.insert(0, os.path.join(EXP_DIR, '.chinese'))
    from chinese_font import setup_chinese_font
    setup_chinese_font(save_dir=os.path.join(EXP_DIR, '.chinese'))
    CN = True
except Exception:
    plt.rcParams['axes.unicode_minus'] = False

def t(cn, en):
    return cn if CN else en

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def save_fig(fig, name):
    for d in OUT_DIRS:
        fig.savefig(os.path.join(d, name), dpi=140, bbox_inches='tight')
    print('已保存', name, '->', OUT_DIRS)

## ④ 参数冻结表（三组共用，唯一差异是 maxit 与核形式）

| 参数 | 值 | 说明 / 公式对应 |
|---|---|---|
| SEED | 42 | 每组开跑前重置，保证三组使用**完全相同**的噪声实现 |
| BSNR | 40 dB | 与现实验、官方 lab2 一致 |
| 模糊核 | 5×5 uniform（循环卷积） | 前向算子 $A$ |
| ε | (5/255)² | 去噪器噪声水平 / 先验正则 |
| α, L_net | 1, 1 | RealSN-DnCNN 谱归一化保证 Lipschitz=1 |
| δ | 0.99/(L_net/ε+L_y) | 投影 PPnP-ULA 指导上界（简化核沿用同一步长） |
| 指标记录间隔 | 每 10 次（burn-in 后） | Welford 统计量仍**每步更新**，仅画图指标降采样 |

In [ ]:
# ④ 参数冻结
SEED        = 42
BSNRdb      = 40
KERNEL_LEN  = [5, 5]
TYPE_BLUR   = 'uniform'
ALPHA       = 1.0
EPS         = (5.0/255.0)**2
L_NET       = 1.0
DELTA_FRAC  = 0.99
METRIC_STRIDE = 10

# 三组对照配置（thinning 与现实验/官方口径一致）
RUN_ORDER = ['A_short500_simple', 'B_long10k_simple', 'C_long10k_ppnp']
RUNS = {
    'A_short500_simple':  dict(maxit=500,   burnin=50,  n_samples=50,   kernel='simple',
                               name=t('A 短链500·简化核（复现现实验）', 'A short 500 · simplified (current exp.)')),
    'B_long10k_simple':   dict(maxit=10000, burnin=500, n_samples=2000, kernel='simple',
                               name=t('B 长链10k·简化核', 'B long 10k · simplified')),
    'C_long10k_ppnp':     dict(maxit=10000, burnin=500, n_samples=2000, kernel='ppnp',
                               name=t('C 长链10k·投影PPnP（官方）', 'C long 10k · projected PPnP (official)')),
}
for _c in RUNS.values():
    _c['thinning'] = max(1, (_c['maxit'] - _c['burnin']) // _c['n_samples'])
print(json.dumps({k: {kk: vv for kk, vv in v.items() if kk != 'name'} for k, v in RUNS.items()},
                 ensure_ascii=False, indent=2))

In [ ]:
# ⑤ 前向模型 A 与含噪观测 y（只构造一次，三组共用）
torch.manual_seed(SEED)
np.random.seed(SEED)

im = np.array(Image.open(os.path.join(EXP_DIR, 'cman.png')))
x = torch.Tensor(im / 255.).to(device)
A, AT, AAT_norm = blur_operators(KERNEL_LEN, [im.shape[0], im.shape[1]], TYPE_BLUR, device)

y0 = A(x)
sigma = torch.linalg.matrix_norm(A(x) - torch.mean(A(x)), ord='fro') \
        / math.sqrt(torch.numel(x) * 10 ** (BSNRdb / 10))
y = y0 + sigma * torch.randn_like(x)

gradf = lambda X: AT(A(X) - y) / sigma ** 2
L_y = AAT_norm / sigma ** 2

# —— 三种核的步长（公式 → 代码映射）——
DELTA_PPNP = DELTA_FRAC * 1.0 / (L_NET / EPS + L_y)                       # simple / ppnp 共用
_max_lam_full = 1.0 / ((2.0 * ALPHA * L_NET) / EPS + 4.0 * L_y)          # 完整未投影 PnP-ULA
LAM_FULL  = DELTA_FRAC * _max_lam_full
DELTA_FULL = DELTA_FRAC * (1.0 / 3.0) / (ALPHA * L_NET / EPS + L_y + 1.0 / LAM_FULL)

C_upper = torch.tensor(1.0, device=device)
C_lower = torch.tensor(0.0, device=device)
projbox = lambda v: torch.clamp(v, min=C_lower, max=C_upper)

print(f'图像 {im.shape} | sigma = {sigma.item():.6f}（现实验 JSON = 0.002484）')
print(f'AAT_norm = {AAT_norm:.4f} | L_y = {L_y:.1f} | eps = {EPS:.8f}')
print(f'delta(simple/ppnp) = {DELTA_PPNP:.3e} | delta(完整PnP-ULA) = {DELTA_FULL:.3e}')
print(f'初始 PSNR = {PSNR(x, y):.2f} dB | NRMSE = {NRMSE(x, y):.4f} | SSIM = {SSIM(x, y):.4f}')

# —— 附：CPU 备选 MYULA 的【修正后】严格步长（复核报告第 3.4 节）——
# 现实验 4.7-1.py 取 delta=1/L_y，越过严格上界；严格 MYULA 应为:
lam_myula = 1.0 / (2.0 * L_y)
delta_myula_strict = (1.0 / 3.0) / (L_y + 1.0 / lam_myula)
print(f'[备选MYULA修正] lambda={lam_myula:.3e}, 严格 delta_max=(1/3)/(L_y+1/lambda)='
      f'{delta_myula_strict:.3e}（旧值 1/L_y = {1.0/L_y:.3e}，大 3 倍）')

In [ ]:
# ⑥ 加载 RealSN-DnCNN（残差网：denoise(x)=x-model(x) 才是去噪图）
model = load_model(MODEL_PATH, device)
@torch.no_grad()
def denoise(v):
    return (v - model(v[None][None])[0][0]).detach()
print('去噪器加载完成：', MODEL_PATH)

## ⑦ Markov 核（公式 → 代码）

- **simple（现实验简化核，无投影、无尾项）**：
  $X_{k+1}=X_k-\delta\nabla f(X_k)+\frac{\alpha\delta}{\varepsilon}(D_\varepsilon(X_k)-X_k)+\sqrt{2\delta}\,Z_k$
- **ppnp（官方 lab2 默认，硬投影到 $C=[0,1]^n$）**：
  $X_{k+1}=\Pi_C\!\bigl(X_k-\delta\nabla f+\frac{\alpha\delta}{\varepsilon}(D-X_k)+\sqrt{2\delta}Z_k\bigr)$
- **pnp_full（完整未投影 PnP-ULA，本次不跑，仅保留参考实现）**：
  额外加 $\frac{\delta}{\lambda}(\Pi_C(X_k)-X_k)$，且 $\delta$ 取严格上界 $(1/3)L_{total}^{-1}$。

In [ ]:
# ⑦ Markov 核与采样器
def markov_step(X, kind, delta, lam):
    z = math.sqrt(2.0 * delta) * torch.randn_like(X)
    drift_pnp = -delta * gradf(X) + ALPHA * delta / EPS * (denoise(X) - X)
    if kind == 'ppnp':
        return projbox(X + drift_pnp + z)
    if kind == 'pnp_full':
        return X + drift_pnp + delta / lam * (projbox(X) - X) + z
    if kind == 'simple':
        return X + drift_pnp + z
    raise ValueError(kind)

def run_chain(cfg):
    # 每组重置种子 → 三组的含噪观测、初值、随机数序列完全一致，只有长度/核不同
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    kind = cfg['kernel']
    maxit, burnin, thin = cfg['maxit'], cfg['burnin'], cfg['thinning']
    if kind == 'pnp_full':
        delta, lam = DELTA_FULL, LAM_FULL
    else:
        delta, lam = DELTA_PPNP, None

    X = y.clone()
    wm = None
    MC_X = []
    curves = dict(it=[], psnr=[], ssim=[], nrmse=[])
    count = 0
    t0 = time.time()
    for it in range(maxit):
        X = markov_step(X, kind, delta, lam)
        if it == burnin:
            wm = welford(X)
        elif it > burnin:
            wm.update(X)
            if (it - burnin) % METRIC_STRIDE == 0:
                m = wm.get_mean()
                curves['it'].append(it)
                curves['psnr'].append(float(PSNR(x, m)))
                curves['ssim'].append(float(SSIM(x, m)))
                curves['nrmse'].append(float(NRMSE(x, m)))
            if count == thin - 1:
                MC_X.append(X.detach().cpu().numpy().astype(np.float32))
                count = 0
            else:
                count += 1
        if (it + 1) % 2000 == 0:
            print(f'  {cfg["name"]}: iter {it+1}/{maxit} ({time.time()-t0:.0f}s)')

    MC_X = np.stack(MC_X, axis=0)
    return dict(mean=wm.get_mean().detach().cpu().numpy(),
                var=wm.get_var().detach().cpu().numpy(),
                MC=MC_X, curves=curves,
                elapsed=time.time() - t0, delta=delta,
                final_psnr=float(PSNR(x, wm.get_mean())),
                final_ssim=float(SSIM(x, wm.get_mean())),
                final_nrmse=float(NRMSE(x, wm.get_mean())))
print('采样器定义完成')

In [ ]:
# ⑧ 顺序执行三组对照（Colab GPU 总计约 17 分钟，请耐心等待）
RES = {}
for tag in RUN_ORDER:
    print('\n>>> 运行', RUNS[tag]['name'])
    RES[tag] = run_chain(RUNS[tag])
    r = RES[tag]
    print(f'    耗时 {r["elapsed"]/60:.1f} min | 存样 {r["MC"].shape[0]} | '
          f'PSNR {r["final_psnr"]:.2f} | SSIM {r["final_ssim"]:.4f} | NRMSE {r["final_nrmse"]:.4f}')

In [ ]:
# ⑨ 定量对比表 + 自动判定（参考值来自官方 lab2_PnP_sol.ipynb 的实际输出）
REF = dict(psnr=33.22, ssim=0.9212, nrmse=0.0309,
           init_psnr=21.61, init_nrmse=0.1174, init_ssim=0.6957)
CUR = dict(psnr=28.14, ssim=0.8492, nrmse=0.0555)   # 现实验 results_summary.json

def ess_triple(MC):
    v = MC.reshape(len(MC), -1)
    var_sp = np.var(v, axis=0)
    tr_slow = v[:, np.argmax(var_sp)]
    tr_fast = v[:, np.argmin(var_sp)]
    tr_med  = v[:, np.argsort(var_sp)[len(var_sp) // 2]]
    return (tr_med, tr_slow, tr_fast), tuple(float(arviz.ess(tr_)) for tr_ in (tr_med, tr_slow, tr_fast))

ESS = {tag: ess_triple(RES[tag]['MC'])[1] for tag in RUN_ORDER}

# 平坦区掩膜（真值局部梯度最小的 40% 像素：天空/草地）
_gx, _gy = np.diff(x.cpu().numpy(), axis=1, prepend=0), np.diff(x.cpu().numpy(), axis=0, prepend=0)
_grad = np.sqrt(_gx**2 + _gy**2)
FLAT = _grad <= np.quantile(_grad, 0.40)
flat_std = {tag: float(np.median(np.sqrt(RES[tag]['var'])[FLAT])) for tag in RUN_ORDER}

def plateau_gap(curve):
    p = np.asarray(curve['psnr'])
    k = max(5, len(p) // 10)
    return abs(p[-k:].mean() - p[-2*k:-k].mean())

checks = [
 ('A 组复现现实验 JSON（28.14 ±0.3 dB）',
  abs(RES['A_short500_simple']['final_psnr'] - CUR['psnr']) < 0.3,
  f"A={RES['A_short500_simple']['final_psnr']:.2f} dB"),
 ('C 组达到官方参考 PSNR 33.22（±0.6 dB）',
  abs(RES['C_long10k_ppnp']['final_psnr'] - REF['psnr']) < 0.6,
  f"C={RES['C_long10k_ppnp']['final_psnr']:.2f} dB"),
 ('C 组达到官方参考 SSIM 0.9212（±0.01）',
  abs(RES['C_long10k_ppnp']['final_ssim'] - REF['ssim']) < 0.01,
  f"C={RES['C_long10k_ppnp']['final_ssim']:.4f}"),
 ('C 组达到官方参考 NRMSE 0.0309（±0.004）',
  abs(RES['C_long10k_ppnp']['final_nrmse'] - REF['nrmse']) < 0.004,
  f"C={RES['C_long10k_ppnp']['final_nrmse']:.4f}"),
 ('长链比短链 PSNR 提升 > 3 dB（500 步确实未收敛）',
  RES['C_long10k_ppnp']['final_psnr'] - RES['A_short500_simple']['final_psnr'] > 3.0,
  f"提升 {RES['C_long10k_ppnp']['final_psnr']-RES['A_short500_simple']['final_psnr']:.2f} dB"),
 ('长链 C 末段进入平台（最后10%均值变动 < 0.15 dB）',
  plateau_gap(RES['C_long10k_ppnp']['curves']) < 0.15,
  f"变动 {plateau_gap(RES['C_long10k_ppnp']['curves']):.3f} dB"),
 ('短链 A 末端仍在爬坡（末10%变动 > 0.15 dB）',
  plateau_gap(RES['A_short500_simple']['curves']) > 0.15,
  f"变动 {plateau_gap(RES['A_short500_simple']['curves']):.3f} dB"),
 ('平坦区不确定性：长链中位标准差 < 短链的 60%',
  flat_std['C_long10k_ppnp'] < 0.6 * flat_std['A_short500_simple'],
  f"A={flat_std['A_short500_simple']:.4f} → C={flat_std['C_long10k_ppnp']:.4f}"),
 ('ESS：长链中速分量 > 15（短链仅 4.35）',
  ESS['C_long10k_ppnp'][0] > 15,
  f"ESS_med A={ESS['A_short500_simple'][0]:.1f}, C={ESS['C_long10k_ppnp'][0]:.1f}"),
]

print('\n======== 最终指标 ========')
header = f"{'组别':<22}{'PSNR(dB)':>10}{'SSIM':>9}{'NRMSE':>9}{'ESS_med':>9}{'耗时(min)':>10}"
print(header)
for tag in RUN_ORDER:
    r = RES[tag]
    print(f"{tag:<22}{r['final_psnr']:>10.2f}{r['final_ssim']:>9.4f}{r['final_nrmse']:>9.4f}"
          f"{ESS[tag][0]:>9.1f}{r['elapsed']/60:>10.1f}")
print(f"{'官方参考lab2':<20}{REF['psnr']:>10.2f}{REF['ssim']:>9.4f}{REF['nrmse']:>9.4f}")

print('\n======== 自动判定 ========')
all_pass = True
for name, ok, detail in checks:
    all_pass &= ok
    print(('[PASS] ' if ok else '[FAIL] ') + name + f'   （{detail}）')
print('\n总判定:', '全部通过 —— 复核结论成立' if all_pass else '存在未通过项，需检查输出')

summary = dict(
    setup=dict(seed=SEED, BSNR=BSNRdb, kernel_len=KERNEL_LEN, eps=EPS, sigma=float(sigma.item()),
               L_y=float(L_y), delta=float(DELTA_PPNP)),
    reference=REF, current_experiment_json=CUR,
    runs={tag: dict(config={k: v for k, v in RUNS[tag].items() if k != 'name'},
                    psnr=RES[tag]['final_psnr'], ssim=RES[tag]['final_ssim'],
                    nrmse=RES[tag]['final_nrmse'],
                    ess_med=ESS[tag][0], ess_slow=ESS[tag][1], ess_fast=ESS[tag][2],
                    flat_region_median_std=flat_std[tag],
                    elapsed_sec=RES[tag]['elapsed'], stored_samples=int(RES[tag]['MC'].shape[0]))
           for tag in RUN_ORDER},
    checks=[dict(name=n, passed=bool(ok), detail=d) for n, ok, d in checks],
    all_passed=bool(all_pass))
for d in OUT_DIRS:
    with open(os.path.join(d, 'verify_summary.json'), 'w', encoding='utf-8') as fp:
        json.dump(summary, fp, ensure_ascii=False, indent=2)
print('\nverify_summary.json 已保存至', OUT_DIRS)

In [ ]:
# ⑩ PSNR / SSIM / NRMSE 收敛曲线叠加（横轴 = 原始迭代步数）
fig, axes = plt.subplots(1, 3, figsize=(17, 4.2))
colors = dict(A_short500_simple='tab:red', B_long10k_simple='tab:blue', C_long10k_ppnp='tab:green')
for tag in RUN_ORDER:
    c = RES[tag]['curves']
    axes[0].plot(c['it'], c['psnr'],  color=colors[tag], lw=1.4, label=RUNS[tag]['name'])
    axes[1].plot(c['it'], c['ssim'],  color=colors[tag], lw=1.4)
    axes[2].plot(c['it'], c['nrmse'], color=colors[tag], lw=1.4)
axes[0].axhline(REF['psnr'], ls='--', c='k', lw=1, label=t('官方参考 33.22', 'ref 33.22'))
axes[0].axvline(500, ls=':', c='gray', lw=1)
axes[0].annotate(t('现实验终点≈官方burn-in边界', 'current end ≈ ref burn-in'),
                 xy=(500, 28.3), xytext=(1200, 26.5), arrowprops=dict(arrowstyle='->'))
axes[0].set_ylabel('PSNR (dB)'); axes[1].set_ylabel('SSIM'); axes[2].set_ylabel('NRMSE')
for ax in axes:
    ax.set_xlabel(t('迭代步数', 'iteration')); ax.grid(alpha=0.3); ax.set_xlim(0, 10000)
axes[0].legend(fontsize=8, loc='lower right')
fig.suptitle(t('PnP-ULA 链长对照：累积后验均值的重建质量', 'PnP-ULA chain-length comparison'))
plt.tight_layout()
save_fig(fig, 'comparison_curves.png')
plt.show()

In [ ]:
# ⑪ 后验均值与不确定性图并排（标准差图三组共用同一色标，便于直接比较）
stds = {tag: np.sqrt(RES[tag]['var']) for tag in RUN_ORDER}
vmax = np.percentile(stds['A_short500_simple'], 99)
fig, axes = plt.subplots(2, 4, figsize=(17, 8.5))
axes[0,0].imshow(x.cpu().numpy(), cmap='gray'); axes[0,0].set_title(t('真值', 'ground truth'))
axes[0,1].imshow(y.cpu().numpy(), cmap='gray'); axes[0,1].set_title(t('含噪模糊观测 y', 'observation y'))
for ax, tag in zip([axes[0,2], axes[0,3], axes[1,0]],
                   ['A_short500_simple','B_long10k_simple','C_long10k_ppnp']):
    ax.imshow(RES[tag]['mean'], cmap='gray')
    ax.set_title(t('后验均值', 'post. mean') + f' · {tag.split("_")[0]} · '
                 + f'PSNR {RES[tag]["final_psnr"]:.2f}')
im_h = None
for ax, tag in zip([axes[1,1], axes[1,2], axes[1,3]],
                   ['A_short500_simple','B_long10k_simple','C_long10k_ppnp']):
    im_h = ax.imshow(stds[tag], cmap='hot', vmin=0, vmax=vmax)
    ax.set_title(t('后验标准差（统一色标）', 'post. std (shared scale)') + f' · {tag.split("_")[0]}')
for ax in axes.flat:
    ax.axis('off')
fig.colorbar(im_h, ax=axes[1,1:].tolist(), fraction=0.02, pad=0.01)
fig.suptitle(t('上排：真值 / y / 三组后验均值；下排：后验标准差 A→B→C',
               'Row1: truth, y, posterior means; Row2: posterior std A->B->C'), y=1.02)
plt.tight_layout()
save_fig(fig, 'comparison_images.png')
plt.show()

In [ ]:
# ⑫ 三组各自的中速/最慢/最快像素分量 ACF（ESS 见图题；1 个滞后 = thinning 个原始迭代）
for tag in RUN_ORDER:
    (tr_med, tr_slow, tr_fast), (e_med, e_slow, e_fast) = ess_triple(RES[tag]['MC'])
    nlags = min(100, RES[tag]['MC'].shape[0] - 1)
    fig, axs = plt.subplots(1, 3, figsize=(15, 3.8))
    for ax, tr, e, lab in zip(
            axs, (tr_med, tr_slow, tr_fast), (e_med, e_slow, e_fast),
            (t('中速分量', 'median'), t('最慢分量', 'slowest'), t('最快分量', 'fastest'))):
        plot_acf(tr, ax=ax, lags=nlags, alpha=None)
        ax.set_title(f'{lab} ACF  ESS={e:.1f}')
        ax.set_xlabel(t(f'滞后 k（1 k = {RUNS[tag]["thinning"]} 次原始迭代）',
                        f'lag k (1 k = {RUNS[tag]["thinning"]} iters)'))
    fig.suptitle(RUNS[tag]['name'], y=1.03)
    plt.tight_layout()
    save_fig(fig, f'acf_{tag}.png')
    plt.show()

## ⑬ 结果判读

- **若 ⑨ 全部 PASS**：证明现实验 28.14 dB 只是 500 步瞬态，跑满 10 000 步即可达到官方 33.2 dB；
  后验方差图平坦区应由“红噪声场”变为接近全黑；ACF 最慢分量的余弦状漂移应消失、ESS 显著增大。
  这说明“低 ESS / 高方差”不是 PnP-ULA 的固有属性，而是迭代预算被压缩 20 倍所致。
- 长链 B（简化核）与 C（投影核）指标应非常接近（差值通常 < 0.2 dB），说明无投影简化在本例
  数值上确实稳定、影响很小；但严格的未投影 PnP-ULA 应使用 $\delta<(1/3)L_{total}^{-1}$（本脚本已提供该步长数值）。
- 真正用于**不确定性量化**时，按官方说明需要 5×10⁵–10⁶ 次迭代；10⁴ 次仅用于演示与收敛判断。
- 产物：`comparison_curves.png`、`comparison_images.png`、`acf_*.png`、`verify_summary.json`。